# CLIP Multimodal Model

**CLIP (Contrastive Language-Image Pretraining)** by OpenAI learns a joint embedding space for text and images. Images and their text descriptions are pulled close together; non-matching pairs are pushed apart.

Applications:
- **Zero-shot image classification** — classify into any labels without task-specific training
- **Image-text similarity** — score how well a caption matches an image
- **Image retrieval** — find the most relevant image for a text query

## Setup

In [ ]:
!pip install transformers datasets sentence-transformers pillow -q

## Imports

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests
from io import BytesIO
import torch
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {device}")

## 1. Load CLIP

In [ ]:
MODEL_ID = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(MODEL_ID).to(device)
clip_processor = CLIPProcessor.from_pretrained(MODEL_ID)
print(f"CLIP loaded: {MODEL_ID}")

## 2. Zero-Shot Image Classification

Provide candidate text labels — CLIP picks the one that best matches the image, with no task-specific training.

In [ ]:
HEADERS = {"User-Agent": "HuggingFace-Lab/1.0 (educational notebook; https://github.com)"}

def load_image(url):
    r = requests.get(url, timeout=10, headers=HEADERS)
    r.raise_for_status()
    return Image.open(BytesIO(r.content)).convert("RGB")

def zero_shot_classify(image, candidate_labels):
    inputs = clip_processor(
        text=candidate_labels, images=image, return_tensors="pt", padding=True
    ).to(device)
    with torch.no_grad():
        outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)[0].cpu().numpy()
    return dict(zip(candidate_labels, probs.tolist()))

# Test images
test_cases = [
    {
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/320px-Cat_November_2010-1a.jpg",
        "labels": ["a photo of a cat", "a photo of a dog", "a photo of a bird", "a photo of a car"],
    },
    {
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg",
        "labels": ["a photo of a cat", "a photo of a dog", "a photo of a bird", "a photo of a car"],
    },
]

fig, axes = plt.subplots(len(test_cases), 2, figsize=(12, 4 * len(test_cases)))

for row, tc in enumerate(test_cases):
    try:
        img = load_image(tc["url"])
        scores = zero_shot_classify(img, tc["labels"])

        axes[row][0].imshow(img); axes[row][0].axis("off")
        axes[row][0].set_title(f"Top: {max(scores, key=scores.get)}")

        axes[row][1].barh(list(scores.keys()), list(scores.values()), color="steelblue")
        axes[row][1].set_xlim(0, 1); axes[row][1].set_xlabel("Probability")
        axes[row][1].set_title("Zero-Shot Classification")
    except Exception as e:
        print(f"Load error: {e}")
        axes[row][0].text(0.5, 0.5, "load error", ha="center"); axes[row][0].axis("off")

plt.tight_layout(); plt.show()

## 3. Image-Text Similarity Matrix

Compute a full cross-modal similarity matrix between a set of images and a set of captions.

In [ ]:
image_data = {
    "cat": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/320px-Cat_November_2010-1a.jpg",
    "dog": "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg",
}
captions = [
    "a cute cat sitting",
    "a dog looking at the camera",
    "a sports car on the road",
    "a bowl of pasta",
]

loaded_images, img_names = [], []
for name, url in image_data.items():
    try:
        loaded_images.append(load_image(url))
        img_names.append(name)
    except Exception:
        pass

if loaded_images:
    inputs = clip_processor(text=captions, images=loaded_images, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = clip_model(**inputs)
    sim_matrix = outputs.logits_per_image.softmax(dim=-1).cpu().numpy()

    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(sim_matrix, cmap="Blues", vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(captions))); ax.set_xticklabels(captions, rotation=30, ha="right", fontsize=8)
    ax.set_yticks(range(len(img_names))); ax.set_yticklabels(img_names)
    ax.set_title("CLIP Image-Text Similarity Matrix")
    for i in range(len(img_names)):
        for j in range(len(captions)):
            ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha="center", va="center", fontsize=9)
    plt.tight_layout(); plt.show()